# GNote Audit Report: Placebos + MVP Violations

- Date: 2026-03-13
- Scope scanned: `lib/`, `schema/`, `supabase/`
- Focus: placebo behavior (looks implemented but not reliable) and MVP architecture violations.

## High Severity Findings

### 1) Inconsistent 9am lock logic across layers (behavior bug)
- Status: **Fixed in this pass**.
- Router lock check now uses the same Harare-based logic as `Daily3Notifier`.
- Result: anchor gating and Daily3 locking now align.

References:
- `lib/services/providers.dart:310`
- `lib/core/router.dart:51`

### 2) Sync layer silently swallows failures (sync success can be placebo)
- Push/pull operations catch and ignore exceptions in many paths.
- UI state can appear successful even when cloud sync failed.
- No surfaced retry state or user-visible failure signal.

References:
- `lib/services/sync.dart:40`
- `lib/services/sync.dart:92`
- `lib/services/providers.dart:271`

## Medium Severity Findings

### 3) Notification permission flow is not wired (possible placebo reminders)
- Status: **Fixed in this pass**.
- Added permission guard methods and switched auth success flow to call
  `ensurePermissionAndScheduleAll(...)`.
- Habit reminder schedule now uses persisted hour/minute when available.

References:
- `lib/services/notification_service.dart:75`
- `lib/pages/login_page.dart:50`

### 4) Habit reminder time is not persisted (UI setting can be placebo)
- Status: **Fixed in this pass**.
- Added persistence in local meta box (`hour` + `minute`).
- Profile now loads stored value on init and saves user updates.
- Notification update now returns success/failure; UI feedback reflects actual outcome.

References:
- `lib/services/local_db.dart:202`
- `lib/pages/profile_page.dart:24`
- `lib/pages/profile_page.dart:55`

## Remediation Applied (This Pass)
- Lock consistency fix: `lib/core/router.dart` now uses Harare lock-time computation.
- Notification flow fix: `NotificationService.ensurePermission*` methods added.
- Auth success flows updated to permission-aware scheduling:
  - `lib/pages/login_page.dart`
  - `lib/pages/signup_page.dart`
  - `lib/pages/verify_opt_page.dart`
- Habit reminder persistence added in local DB and wired in profile:
  - `lib/services/local_db.dart`
  - `lib/pages/profile_page.dart`

### 5) MVP boundary leak: page directly uses local DB
- `AnchorPage` directly depends on `LocalDb` and handles draft persistence logic.
- In strict MVP, this belongs in presenter/notifier/service layer.

References:
- `lib/pages/anchor_page.dart:21`
- `lib/pages/anchor_page.dart:40`

### 6) MVP boundary leak: router reads infra/data directly
- Router guard checks Supabase session and local DB state directly.
- This bypasses presenter/view-state abstraction.

References:
- `lib/core/router.dart:40`
- `lib/core/router.dart:53`

## Low Severity Findings

### 7) Dead/placeholder feature surface: `selections`
- Box/table/constants exist, but app logic does not actively use this feature.
- This is scope overhead for MVP unless intentionally staged for near-term rollout.

References:
- `lib/core/constants.dart:351`
- `lib/main.dart:34`
- `lib/services/local_db.dart:19`
- `supabase/schema.sql:137`

## Validation Run
- `flutter analyze` completed with: **No issues found**.
- Note: static analysis passing does not invalidate behavioral/placebo risks above.